# 0.6 From complex fields to diffraction orders

[0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) determined a field from an absorbing material structure. Choose a slice through that field. Can we rebuild it by adding diffraction orders? We will find the complex weight of each order, then move the observation plane to see which components carry radiation away and which decay near the grating.

## Motivation and prerequisites

Recall $U=|U|e^{i\phi}$, field superposition, and the physical setup of [0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb). From [0.4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb), recall that homogeneous propagation acts on individual wave components. You need the meaning of a complex amplitude, not the details of the FMM solver.

First compare a field with the sum of its components. Then follow one component as the plane moves, and finally calculate its contribution to power. The Fourier integral and complex square roots explain these steps mathematically; they can be revisited after the plots.

Compare the reconstructed field with its samples, move the output plane, and follow orders 0, 2 and 3. Distinguish a phase change from amplitude decay before studying the formulas. The later chapters develop the wavevector, evanescent-field and intensity calculations in more detail.

[0.7](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb) compares these physical coefficients with those of an ideal binary amplitude mask at the same reference plane.

:::{note} Model and wavelength conventions

**Medium and wavelength convention.** The inherited scattering configuration takes **vacuum** wavelength $\lambda_0$, despite the code field name `wavelength`. Incident and output media are air ($n_i=n_t=1$), so their wavelengths numerically equal $\lambda_0$ in this model. The absorbing layer has its own spatially varying complex index. Coefficients and normal flux refer to the stated exterior reference planes. See the [course medium convention](0_0_Foundations_of_Scalar_Optics.ipynb#medium-and-wavelength).
:::

Run all cells here in a fresh kernel; running [0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) first is unnecessary. Setup computes one absorbing solution and all views reuse it. We retain $e^{-i\omega t}$, normal incidence toward $-z$, unit incident amplitude and phase zero at $z=0$.


For a later synthesis of coefficient magnitude, phase and normal power, see [7.4: diffraction kinematics and complex coefficients](7_4_Diffraction_Kinematics_and_Complex_Order_Coefficients.ipynb).

:::{note} Interactive examples
Click the **power button** near the top of the page to activate the examples. Binder supplies Python in your browser; no local installation is needed. Wait for the session to connect and the scattering calculation to finish. Use **Slice z** for reconstruction, then **Output plane z_ref** and **Follow order m** for the linked output, evolution and efficiency views. These controls inspect the same solved field. Use **Reset this view** to restore the starting values. The text, equations and saved figures can be read without activating a session.
:::

For local use, download the {download}`numerical explorer <downloads/grating_physics_numerical_explorer.zip>` or the complete {download}`worked-example scripts <downloads/grating_physics_scripts.zip>` with their shared helpers.

In [ ]:
from pathlib import Path
import sys
import importlib
from IPython.display import display

_support_dir = next((base for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))

# Release current explorers and any remaining objects from earlier notebook versions.
for _name in ('absorber_comparison', 'binary_masks', 'grating_comparison'):
    if _name in globals():
        globals()[_name].close()
if 'grating_views' in globals():
    for previous in grating_views.values():
        previous.close()

# Refresh dependencies before their consumers, including a cached pre-absorber solver.
# Import and reload one module at a time: importing all first can fail on new symbols.
for _module_name in ('scalar_grating', 'binary_transmission', 'thin_grating',
                     'absorbing_grating', 'scalar_grating_view', 'amplitude_grating',
                     'grating_comparison_view', 'absorbing_grating_view'):
    importlib.reload(importlib.import_module(f'wave_optics.{_module_name}'))

import wave_optics.absorbing_grating as absorbing_model
import wave_optics.scalar_grating_view as grating_view
import wave_optics.amplitude_grating as amplitude_model
import wave_optics.absorbing_grating_view as absorbing_view

# One shared physical absorber: centred air opening, air half-spaces.
# Compute 49 harmonics; display only orders -7,...,+7 in coefficient plots.
grating_config = absorbing_model.AbsorbingGratingConfiguration(M=24)
display_order_limit = 7
grating_solution = absorbing_model.solve_absorbing_grating(grating_config)
grating_data = grating_view.ScalarGratingData(grating_solution)
grating_views = {}

def show_grating(stage):
    if stage in grating_views:
        grating_views[stage].close()
    grating_views[stage] = grating_view.ScalarGratingExplorer(
        stage, grating_data, display_order_limit=display_order_limit)
    display(grating_views[stage].ui)


### The field we are decomposing

**Basis reminder:** selecting propagating orders for output is different from restricting the scattering solve to those labels. [0.5’s numerical example](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb#can-we-solve-using-only-propagating-orders) shows why evanescent harmonics can change the propagating coefficients.


The grating is unchanged: absorbing ridges with index $1.5+0.8i$ occupy $-h<z<0$, separated by centred air openings. Air lies above and below. The period is $d=1.50\,\mu\mathrm m$, opening width $W_{\mathrm{open}}=0.75\,\mu\mathrm m$, thickness $h=0.30\,\mu\mathrm m$, and vacuum wavelength $\lambda_0=0.633\,\mu\mathrm m$.

The model supplies the full complex field, not only an intensity image. Above the layer it includes incident and reflected light; inside it includes material coupling; below it contains outgoing transmitted contributions. We will first inspect a slice anywhere in that field, then restrict the reference plane to the transmitted air region.

Write the transverse basis as $e^{imGx}$, where $G=2\pi/d$ and integer $m$ labels a harmonic. Its complex weight $F_m(z)$ tells us how much of that spatial variation is present at height $z$. The output notation $C_m$ will refer to the same weight evaluated at a specified transmitted reference plane.

**Computation versus display:** $M=24$ retains 49 harmonics. Coefficient plots and selectors show only $-7\leq m\leq7$ for clarity; field sums, numerical checks and power totals still use all 49. An order can be retained in the solution even when it is evanescent.


## 1. Can the harmonics reproduce the real-space field?

**Predict:** if we keep every harmonic used by the solver, should summing them produce a different field?

**Explore:** move the slice through the incident region, the grating and the output region. The cyan line selects $z$; the bottom plots show $F_m(z)$. Compare the field curves with the reconstruction markers.

In [ ]:
show_grating('reconstruction')

At any chosen $z$, the same complex weights give

$$\boxed{U(x,z)=\sum_m F_m(z)e^{ik_{x,m}x}.}$$

The curves sample the calculated field directly. The markers rebuild it by adding the extracted harmonics. Both use **the same truncated Fourier-modal solution**, so agreement checks that we have represented that solution consistently. It does not test the scattering calculation against an independent method.

Conversely, Fourier projection over one period extracts the weights:

$$F_m(z)=\frac{1}{d}\int_{-d/2}^{d/2}U(x,z)e^{-imGx}\,dx.$$

The profiles show one period; the map shows two. Coefficient phases use the fixed incident-wave reference. Keeping magnitudes alone would generally lose the phase relationships needed to reconstruct the field.

**Takeaway:** a real-space field and its complete complex harmonic coefficients contain the same information at the chosen plane.


### Checkpoint: can samples recover one complex coefficient?

**Predict:** should projecting the sampled field give the same coefficient as reading the solved harmonics?

**Try it in code:** choose `try_m = 0`, `1`, `2` or `3`, then change `try_distance`. Samples cover one period without repeating its endpoint. Their mean approximates the normalized integral above.


In [ ]:
import numpy as np

try_m = 1
try_distance = 0.20  # Micrometres below the exit.
try_d = grating_config.period
try_z_ref = -grating_config.thickness - try_distance
try_x = np.linspace(-try_d / 2, try_d / 2, 512, endpoint=False)
try_u = grating_solution.reconstruct(try_x, try_z_ref)[0]
try_projected = np.mean(try_u * np.exp(-2j * np.pi * try_m * try_x / try_d))
try_j = int(np.flatnonzero(grating_solution.orders == try_m)[0])
try_direct = grating_solution.at_reference_plane(try_z_ref)[try_j]
print(f"From complex samples: {try_projected:+.8f}")
print(f"From solved harmonics: {try_direct:+.8f}")
print(f"Absolute difference: {abs(try_projected - try_direct):.3e}")


:::{dropdown} Check your prediction


The two routes agree to numerical precision for this sampled, truncated solution. They are two representations of the same field. Replacing `try_u` by its magnitude or intensity would generally give different coefficients and would not test this identity.


:::


## 2. What changes when we move the output reference plane?

We have solved the physical absorber once. Now choose where to inspect its output before comparing it with the ideal mask in [0.7](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb).

**Predict:** does moving an observation plane require changing the boundary conditions at the grating?

**Explore:** start just below the bottom interface, $z_{\mathrm{ref}}=-0.30\,\mu\mathrm m$, then move to $-0.50$, $-1.00$ and $-1.80\,\mu\mathrm m$. More negative $z$ means farther from the grating. Compare the field profiles and the coefficient magnitudes and phases.


In [ ]:
show_grating('reference')

The output reference plane stays in the homogeneous transmitted region, $z_{\mathrm{ref}}\leq-h$. Its coefficients are simply

$$\boxed{C_m(z_{\mathrm{ref}})=F_m(z_{\mathrm{ref}}),\qquad
U(x,z_{\mathrm{ref}})=\sum_m C_m(z_{\mathrm{ref}})e^{imGx}.}$$

**$F_m$ is a function; $C_m$ is its value at a stated plane.** Both are complex amplitudes, not intensities or distinct physical fields. In the outgoing homogeneous transmitted region,

$$\boxed{F_m(z)=C_m(z_{\mathrm{ref}})
 e^{ik_{z,m}(z-z_{\mathrm{ref}})}.}$$

For transmission toward $-z$, $k_{z,m}$ uses the outgoing propagating or decaying branch. At the exit $z_e=-h$, $U_{\mathrm{exit}}(x)=U(x,z_e)$ and $C_m(z_e)=F_m(z_e)$. The backend array `solution.transmission` stores these exit coefficients. A new reference plane changes the coefficient value, not the underlying solution. Inside the patterned layer, the $F_m(z)$ remain coupled and cannot generally be replaced by separate single-exponential propagation factors.

The reference plane is a **mathematical observation surface**. Moving it samples the field already calculated; it introduces no new material boundary and requires no new boundary-value solve. The colour scales and profile limits stay fixed so that field changes can be compared directly.

**Takeaway:** the structure selects the solution; the reference plane selects a slice of it.

- The displayed transverse interval samples an indefinitely repeated field; it does not impose an illuminated aperture.
- Each ideal order extends across the homogeneous output region, rather than occupying a separate narrow beam drawn from one point.
- The coefficient plots describe that extended field at a chosen plane, not a camera measurement of its phase. [Maystre, §2.3](#ref-0-6-maystre) [\[1\]](#ref-0-6-maystre).

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mp}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


## 3. Which orders retain amplitude, and which decay?

In the homogeneous output region, the longitudinal dispersion relation is

$$k_{z,m}^2=k_0^2-k_{x,m}^2.$$

**Predict:** will all coefficients have the same magnitude when the plane moves farther away?

**Explore:** follow $m=3$, then $m=0$ or $2$. Move the shared output-plane slider. The cyan marker locates the same plane as in section 2; the curves show how the selected coefficient evolves from the bottom interface.


In [ ]:
display(grating_views['reference'].evolution_ui)

Define $\beta_m=\sqrt{k_0^2-k_{x,m}^2}$ with nonnegative real or imaginary part. For outgoing transmission toward $-z$, the signed longitudinal wavevector is $k_{z,m}=-\beta_m$.

Starting from the exit coefficient $C_m(-h)$, a downward distance $\Delta\geq0$ gives

$$\boxed{C_m(-h-\Delta)=C_m(-h) e^{i\beta_m\Delta}.}$$

| Dispersion | Type | Evolution away from the grating |
| --- | --- | --- |
| $k_0^2-k_{x,m}^2>0$ | Propagating | $\lvert C_m\rvert=\lvert C_m(-h)\rvert$; phase advances by $\beta_m\Delta$. |
| $k_0^2-k_{x,m}^2=0$ | Grazing | Bounded outgoing solution is constant along $z$; zero normal flux. |
| $k_0^2-k_{x,m}^2<0$ | Evanescent | $\beta_m=i\kappa_m$, so $\lvert C_m\rvert=\lvert C_m(-h)\rvert e^{-\kappa_m\Delta}$. |

This evanescent decay occurs in lossless air and is distinct from material absorption inside the ridges.

For this fixed geometry, orders $-2$ through $+2$ propagate; there are no exactly grazing orders. All other retained orders are evanescent. Their nonzero near fields remain in the calculation.

The phase curves show **unwrapped phase advance** from $z=-h$. The earlier field and coefficient plots use wrapped phase. Phase is hidden for amplitudes below $10^{-6}$; a nearly zero complex number has no useful displayed phase.

Even when individual propagating magnitudes stay fixed, their relative phases change. Their sum can therefore have a changing intensity profile.

**Takeaway:** homogeneous propagation changes each known coefficient by a phase factor or an evanescent decay factor.


## 4. Where does the incident power go?

**Predict:** must all incident power leave as reflection or transmission when the ridges absorb? Does an evanescent coefficient carry outgoing normal power?

**Explore:** move the shared reference plane and compare $|C_m|$, phase relative to the zeroth order, and transmitted scalar efficiency. The upper plots show coefficient magnitude and phase; the lower-left plot shows transmitted power by order. Hollow circles mark zero outgoing normal power. The lower-right bars account for reflection, transmission and absorption using all solved orders.
- Which plots change when the plane moves? 

In [ ]:
display(grating_views['reference'].efficiency_ui)

A coefficient contains magnitude and phase, $C_m=|C_m|e^{i\phi_m}$. Its squared magnitude is not generally its power efficiency. With unit incident amplitude and air on both sides,

$$\eta_m^{(T)}=\frac{\operatorname{Re}\beta_m}{k_0}|C_m|^2,\qquad
\eta_m^{(R)}=\frac{\operatorname{Re}\beta_m}{k_0}|r_m|^2.$$

The longitudinal factor weights flux through a plane normal to $z$. Propagating efficiencies stay fixed as the output plane moves. Evanescent and grazing orders carry zero outgoing normal power, even when their field amplitudes are nonzero.

For the physical absorber,

$$\boxed{R+T+A=1,\qquad R=\sum_m\eta_m^{(R)},\quad T=\sum_m\eta_m^{(T)}.}$$

Here $A$ is calculated **independently from loss inside the material**, rather than defined as the missing outgoing power. At the initial geometry, $R\approx0.06409$, $T\approx0.42515$ and $A\approx0.51076$. Moving the observation plane in air changes none of these totals.

:::{dropdown} Optional: the volume-loss check

For unit incident amplitude,

$$A=\frac{k_0}{d}\int_{-h}^{0}\int_{-d/2}^{d/2}\operatorname{Im}\varepsilon(x)\,|U(x,z)|^2\,dx\,dz.$$

The normal scalar flux is proportional to $\operatorname{Im}(U^*\partial_zU)$; its downward value decreases through the absorbing layer. The volume integral and exterior fluxes independently give $R+T+A\simeq1$.

:::

Flux balance checks consistency, but does not establish spatial convergence. At $M=24$, transmitted coefficients with $|m|\leq4$ differ by at most $2.55\times10^{-5}$ from $M=72$ for this geometry. This is not a bound on every local near-field feature. See [Appendix 1: numerical checks](Appendix_1_Fourier_Modal_Method_for_Scalar_Grating_Scattering.ipynb#fmm-validation-checks).


### Checkpoint: does moving the plane change an order's power?

**Predict:** which changes for $m=1$ and $m=3$: coefficient magnitude, phase, or outgoing normal power?

**Try it in code:** compare these two orders at `try_distance = 0.25` and `0.50` µm. The efficiency factor is the one derived above. Keep the distance nonnegative.


In [ ]:
import numpy as np

try_m = 1  # Then try 3.
try_distance = 0.25
try_j = int(np.flatnonzero(grating_solution.orders == try_m)[0])
try_exit = -grating_config.thickness
try_beta = grating_solution.beta_output[try_j]
try_k0 = 2 * np.pi / grating_config.wavelength
print("z (um)       |C_m|       arg C_m (rad)       efficiency")
for try_z in (try_exit, try_exit - try_distance):
    try_c = grating_solution.at_reference_plane(try_z)[try_j]
    try_eta = try_beta.real / try_k0 * abs(try_c)**2
    try_phase = f"{np.angle(try_c):+.3f}" if abs(try_c) >= 1e-6 else "undefined"
    print(f"{try_z:+.3f}      {abs(try_c):.6f}      {try_phase:>9}          {try_eta:.6f}")


:::{dropdown} Check your prediction


Order 1 propagates: its magnitude and efficiency stay fixed while its phase advances (the printed phase wraps at $\pm\pi$). Order 3 is evanescent: its magnitude decays and its outgoing normal efficiency is zero at both planes. Neither observation changes the material absorption already calculated in the layer.


:::


## Checkpoint: one field, two representations

**Solved complex field → Fourier coefficients at a reference plane → independent propagation in air → reconstructed field and flux.**

1. Can coefficient magnitudes alone reconstruct the field?
2. Does moving the reference plane change the grating or require another scattering solve?
3. How can the intensity profile change while each propagating order keeps its magnitude?
4. Why is evanescent decay in air different from absorption in the ridges?

:::{dropdown} Check your answers

Reconstruction needs relative phases as well as magnitudes. The reference plane only selects a slice of an existing solution. Different propagation phase advances change how the orders add at each position, so the spatial intensity can change even while their individual propagating powers stay fixed. Evanescent fields decay in lossless air without carrying outgoing normal power; absorption dissipates power in the material and is accounted for separately by $A$.

:::

The reconstruction explorer has its own slice control. The output-field, order-evolution and efficiency views share one reference-plane control. After replacing the reference explorer, redisplay its evolution and efficiency cells; rerunning setup closes all previous views. No plot control re-solves the scattering problem.

For further reading on independent components and their propagation, see [TU Delft, Wavefield propagation](#ref-0-6-delft-wavefield) [\[2\]](#ref-0-6-delft-wavefield). The later [evanescent-wave](4_0_Evanescent_Diffraction_Orders.ipynb) and [amplitude/intensity](6_0_Amplitude_and_Intensity_of_a_Scalar_Wave.ipynb) lessons develop these ideas more slowly.

**Next question:** could an ideal 0/1 transmission function predict these complex coefficients? Continue to [0.7: Ideal binary masks versus physical absorbing gratings](0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb).

[Previous: 0.5](0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) · [Chapter index](0_0_Foundations_of_Scalar_Optics.ipynb)


## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete {download}`scripts download <downloads/grating_physics_scripts.zip>` to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. Can samples recover one complex coefficient?

**Script:** [0_6_01_project_complex_coefficient.py](Wave_solver_py_scripts/scripts/chapter_0/0_6_01_project_complex_coefficient.py)

Recover one coefficient by numerical projection.

### 2. What changes when we move the output reference plane?

**Script:** [0_6_02_reference_plane_evolution.py](Wave_solver_py_scripts/scripts/chapter_0/0_6_02_reference_plane_evolution.py)

Advance complex coefficients to a new reference plane.

### 3. Where does the incident power go?

**Script:** [0_6_03_flux_weights.py](Wave_solver_py_scripts/scripts/chapter_0/0_6_03_flux_weights.py)

Distinguish coefficient magnitude squared from normal power.

### 4. One field, two representations

**Script:** [0_6_04_parseval_field_and_orders.py](Wave_solver_py_scripts/scripts/chapter_0/0_6_04_parseval_field_and_orders.py)

Check Parseval and the importance of complex order phase.

**Numerical cutoffs:** the physical worked scripts start at $M=8$ (17 harmonics), whereas this teaching notebook uses $M=24$ (49 harmonics). The physical inputs agree, but numerical values can differ with truncation. To match the notebook, set `M = 24` in the script and retain adequate sampling; verify convergence for the quantity of interest. This does not change either default. See the [cutoff guide](Wave_solver_py_scripts/scripts/INDEX.md#chapter-0-cutoffs).




(refs-0-6)=

## References

(ref-0-6-maystre)=

[1] D. Maystre, “Analytic Properties of Diffraction Gratings,” in *Gratings: Theory and Numeric Applications*, 2nd revisited ed., E. Popov, Ed. Institut Fresnel, 2014, ch. 2. [Online]. Available: [Maystre, §2.3](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=42), [PDF](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=33).

**Reading notes:** Chapter 2 in E. Popov (ed.), *Gratings: Theory and Numeric Applications*. Sections 2.3–2.4 connect periodicity to outgoing orders; §§2.5.1 and 2.5.3 discuss balance relations and energy. The electromagnetic treatment is optional background for our scalar flux convention.

**Access:** Free institutional book chapter; not public domain. **Read for:** Read Sections 2.3–2.5: order expansion and energy balance.

(ref-0-6-delft-wavefield)=

[2] TU Delft, “Wavefield propagation,” *Advanced Optical Imaging*. [Online]. Available: [source](https://qiweb.tudelft.nl/aoi/wavefieldpropagation/wavefieldpropagation/).

**Reading notes:** §3.6. Reviews phase advance and evanescent decay. Translate its positive propagation distance to our downward distance $\Delta=-z_{\mathrm{ref}}-h$.

## Further Reading

(ref-0-6-numpy-126)=

- NumPy developers, “Discrete Fourier Transform,” *NumPy 1.26 Manual*. [Online]. Available: [source](https://numpy.org/doc/1.26/reference/routines.fft.html).

  **Reading notes:** Useful when moving from the one-order projection snippet to an FFT of all coefficients. Retain the complex samples, normalization and transverse-origin phase.

(ref-0-6-phasor)=

- M. del Mar Sánchez-López, I. Moreno, and A. Martínez-García, “Teaching diffraction gratings by means of a phasor analysis,” *Education and Training in Optics and Photonics*, paper EMA1, 2009. [Online]. Available: [source](https://opg.optica.org/abstract.cfm?uri=ETOP-2009-EMA1). doi: 10.1364/ETOP.2009.EMA1.

  **Reading notes:** **Access:** Publisher marks Open Access; not public domain. **Read for:** Read Sections 1–2: binary gratings, phasors and order amplitudes.

## Reference notes

Use these to connect the coefficient plots with their mathematical meaning.

Freely accessible works are not necessarily in the public domain.

**Convention check:** Translate each source's order signs, propagation side, polarization assumptions and amplitude/power normalization to this notebook's stated model.


